# PriViLege (CVPR 2024): reproduction on Colab

Runs the official PriViLege code (branch `colab-fixes` of `AkilaDissanayake/PriViLege`) on Colab.

**Before you start**
1. Runtime -> Change runtime type -> **A100** or **L4** GPU. A T4 (16 GB) will likely run out of memory at batch 128.
2. Put `CUB_200_2011.tgz` and `miniimagenet.tar` (from the CEC Google Drive folder) in `MyDrive/PriViLege/datasets/`.
3. Run cells 1-6 at the start of **every** Colab session. Then run the experiment cells you need.

A run (base session + all incremental sessions) cannot resume, so each seed must finish within one Colab session.

## 1. Drive, GPU and paths
**Why:** Drive keeps datasets, logs and results after the Colab VM is wiped. Training data goes on the fast local disk (`/content/data`).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/PriViLege'   # persistent: datasets/, logs/, results/
DATA  = '/content/data'                      # fast local disk for training data
REPO  = '/content/PriViLege'
for d in ['datasets', 'logs', 'results']:
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
os.makedirs(DATA, exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the patched code
**Why:** the `colab-fixes` branch has the three one-line fixes (pandas 2, miniImageNet map path, unused `open_clip` import). The commit id goes into the results log so every number can be traced to the exact code.

In [ ]:
if not os.path.exists(REPO):
    !git clone -b colab-fixes https://github.com/AkilaDissanayake/PriViLege.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
COMMIT = !git rev-parse --short HEAD
COMMIT = COMMIT[0]
!git log -1 --oneline

## 3. Install only what is missing
**Why:** do **not** install `requirements.txt` (a full Anaconda dump that breaks Colab). Colab already has torch, transformers, pandas and so on. Only timm is pinned to **0.6.13**. It loads the same `vit_base_patch16_224` checkpoint as the authors' 0.6.12, and 0.6.12 itself fails to import on Colab's Python 3.12. timm 0.9 and newer load a different checkpoint under the same name.

In [ ]:
!pip install -q timm==0.6.13
import timm, torch, transformers, pandas
print('timm', timm.__version__, '| torch', torch.__version__,
      '| transformers', transformers.__version__, '| pandas', pandas.__version__)
assert timm.__version__ == '0.6.13', 'Restart the runtime (Runtime -> Restart session) and run cells 1-3 again'
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
print('GPU:', GPU)

## 4. Dataset helpers
**Why:** copying one archive from Drive and extracting it locally takes seconds or minutes. Reading thousands of small JPEGs directly from Drive would make every epoch many times slower. `prepare()` is called automatically by `run()`.

In [ ]:
import shutil

ARCHIVES = {   # dataset -> (archive in Drive, folder after extraction)
    'cifar100':      ('cifar-100-python.tar.gz', 'cifar-100-python'),
    'cub200':        ('CUB_200_2011.tgz',        'CUB_200_2011'),
    'mini_imagenet': ('miniimagenet.tar',        'miniimagenet'),
}

def prepare(dataset):
    archive, folder = ARCHIVES[dataset]
    src = f'{DRIVE}/datasets/{archive}'
    if os.path.isdir(f'{DATA}/{folder}'):
        pass
    elif os.path.exists(src):
        print(f'Extracting {archive} ...')
        get_ipython().system(f'tar -xf "{src}" -C {DATA}')
    elif dataset == 'cifar100':
        print('CIFAR-100 is not in Drive yet; train.py will download it (saved to Drive afterwards).')
    else:
        raise FileNotFoundError(f'Put {archive} in {DRIVE}/datasets/ first.')
    if dataset == 'mini_imagenet' and not os.path.exists(f'{DATA}/miniimagenet/split/train.csv'):
        os.makedirs(f'{DATA}/miniimagenet/split', exist_ok=True)   # same csv files ship with the repo
        get_ipython().system(f'cp {REPO}/data/index_list/mini_imagenet/*.csv {DATA}/miniimagenet/split/')

def save_cifar_to_drive():
    src, dst = f'{DATA}/cifar-100-python.tar.gz', f'{DRIVE}/datasets/cifar-100-python.tar.gz'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst); print('Saved CIFAR-100 archive to Drive')

def check_layout(dataset):
    need = {
        'cifar100':      ['cifar-100-python/train', 'cifar-100-python/test', 'cifar-100-python/meta'],
        'cub200':        ['CUB_200_2011/images.txt', 'CUB_200_2011/train_test_split.txt',
                          'CUB_200_2011/image_class_labels.txt', 'CUB_200_2011/classes.txt',
                          'CUB_200_2011/images'],
        'mini_imagenet': ['miniimagenet/images', 'miniimagenet/split/train.csv', 'miniimagenet/split/test.csv'],
    }[dataset]
    missing = [p for p in need if not os.path.exists(f'{DATA}/{p}')]
    print(dataset, 'OK' if not missing else f'MISSING: {missing}')

## 5. Check the pretrained weights
**Why:** confirms you get the authors' backbone (ImageNet-21k pre-trained, ImageNet-1k fine-tuned AugReg ViT-B/16). It also caches the ViT and BERT downloads, so a download problem shows up now instead of mid-run.

In [ ]:
m = timm.create_model('vit_base_patch16_224', pretrained=True)
print(m.default_cfg['url'])   # expect .../augreg/B_16-i21k-300ep-...--imagenet2012-...res_224.npz
assert 'i21k' in m.default_cfg['url']
del m
from transformers import BertTokenizer, BertModel
BertTokenizer.from_pretrained('bert-base-cased'); BertModel.from_pretrained('bert-base-cased')
print('ViT-B/16 and bert-base-cased ready')

## 6. `run()` and `summarize()`
**Why:** `run()` uses the official `run_script/vit_run_pretrain.py` arguments unchanged, so ~40 runs have no typos. It does three more things:
- streams the log to Drive (`tee`), so you keep it if Colab disconnects;
- copies only `results.txt` to Drive (each run also writes two ~350 MB `.pth` files);
- appends one row to `results/results_log.csv` (dataset, seed, GPU, commit, A_Base / A_Last / A_Avg, minutes).

Every ablation needs its own `out` name, because the save path doesn't include the `-ED`/`-SKD`/`-taskblock` flags.

In [ ]:
import ast, csv, glob, time, statistics as st, collections

LOG_CSV = f'{DRIVE}/results/results_log.csv'

def parse_results(path):
    accs = ast.literal_eval(open(path).read().strip().splitlines()[-1])   # per-session accuracies
    return accs[0], accs[-1], sum(accs) / len(accs)                       # A_Base, A_Last, A_Avg

def run(dataset, seed, out='PriViLege', losses='-ED -SKD', extra='',
        epochs_base=5, epochs_new=3, batch_size=128):
    prepare(dataset)
    log = f'{DRIVE}/logs/{dataset}_{out}_seed{seed}.log'
    cmd = (f'python train.py -project base -dataset {dataset} -base_mode ft_dot '
           f'-new_mode avg_cos -gamma 0.1 -lr_base 2e-4 -lr_new 2e-4 -decay 0.0005 '
           f'-epochs_base {epochs_base} -epochs_new {epochs_new} -schedule Cosine '
           f'-milestones 20 30 45 -gpu 0 -temperature 16 -start_session 0 '
           f'-batch_size_base {batch_size} -seed {seed} -vit -comp_out 1 -LT {losses} {extra} '
           f'-out {out} -dataroot {DATA}')
    print(cmd, '\n')
    t0 = time.time()
    get_ipython().system(f'cd {REPO} && {cmd} 2>&1 | tee {log}')
    minutes = (time.time() - t0) / 60
    if dataset == 'cifar100':
        save_cifar_to_drive()
    get_ipython().system(f"rsync -a --exclude='*.pth' {REPO}/checkpoint/ {DRIVE}/results/")
    hits = [p for p in glob.glob(f'{REPO}/checkpoint/{out}/{dataset}/**/*seed_{seed}-*/results.txt', recursive=True)
            if os.path.getmtime(p) > t0]
    if not hits:
        print(f'\n!! No new results.txt: the run failed. Check the end of {log}')
        return None
    a_base, a_last, a_avg = parse_results(hits[0])
    new_file = not os.path.exists(LOG_CSV)
    with open(LOG_CSV, 'a', newline='') as f:
        w = csv.writer(f)
        if new_file:
            w.writerow(['dataset', 'seed', 'out', 'flags', 'batch', 'gpu', 'commit',
                        'A_Base', 'A_Last', 'A_Avg', 'minutes', 'date'])
        w.writerow([dataset, seed, out, f'{losses} {extra}'.strip(), batch_size, GPU, COMMIT,
                    f'{a_base:.2f}', f'{a_last:.2f}', f'{a_avg:.2f}', f'{minutes:.1f}', time.strftime('%Y-%m-%d %H:%M')])
    print(f'\n== {dataset} seed {seed} ({out}): A_Base {a_base:.2f} | A_Last {a_last:.2f} | A_Avg {a_avg:.2f} | {minutes:.1f} min')
    return a_base, a_last, a_avg

def summarize(out='PriViLege'):
    by = collections.defaultdict(list)
    for p in sorted(glob.glob(f'{DRIVE}/results/{out}/**/results.txt', recursive=True)):
        ds = p.split(f'/{out}/')[1].split('/')[0]
        seed = p.split('seed_')[1].split('-')[0]
        r = parse_results(p)
        by[ds].append(r)
        print(f'{ds:14s} seed {seed}: Base {r[0]:.2f}  Last {r[1]:.2f}  Avg {r[2]:.2f}')
    for ds, v in by.items():
        cols = list(zip(*v))
        fmt = lambda xs: f'{st.mean(xs):.2f} +/- {st.stdev(xs) if len(xs) > 1 else 0:.2f}'
        print(f'== {ds} (n={len(v)}): A_Base {fmt(cols[0])} | A_Last {fmt(cols[1])} | A_Avg {fmt(cols[2])}')

## 7. Smoke test (1 base epoch + 1 epoch per session, ~10-20 min)
**Why:** this catches environment, data and patch problems before you spend time on full runs. The pandas crash fixed in `colab-fixes` would only appear after the base session, so a pass means all 9 sessions run.

It passes if the log shows `Incremental session: [1]` ... `[8]` and the final line prints A_Base / A_Last / A_Avg. Smoke-test numbers are not meaningful.

In [ ]:
run('cifar100', seed=1, out='smoke', epochs_base=1, epochs_new=1)

## 8. Main reproduction runs (paper Table 1)
**Why 5 seeds:** the paper reports mean +/- std over 5 runs. **Why this order:** CIFAR-100 needs no data preparation. CUB200 is quick and is also used for the ablations. miniImageNet has the most data.

Paper targets (A_Base / A_Last / A_Avg):

| Dataset | A_Base | A_Last | A_Avg |
|---|---|---|---|
| CIFAR-100 | 90.88 +/- 0.20 | 86.06 +/- 0.32 | 88.08 +/- 0.20 |
| CUB200 | 82.21 +/- 0.35 | 75.08 +/- 0.52 | 77.50 +/- 0.33 |
| miniImageNet | 96.68 +/- 0.06 | 94.10 +/- 0.13 | 95.27 +/- 0.11 |

If a session disconnects, run only the seeds that are missing from `results_log.csv`.

In [ ]:
for seed in [1, 2, 3, 4, 5]:
    run('cifar100', seed)

In [ ]:
check_layout('cub200') if os.path.isdir(f'{DATA}/CUB_200_2011') else prepare('cub200')
for seed in [1, 2, 3, 4, 5]:
    run('cub200', seed)

In [ ]:
prepare('mini_imagenet'); check_layout('mini_imagenet')
for seed in [1, 2, 3, 4, 5]:
    run('mini_imagenet', seed)

## 9. Summary vs. the paper
**Why:** computes A_Base, A_Last and A_Avg exactly as the paper defines them, from the files the code writes. The std is the sample std; the paper doesn't say which it uses.

In [ ]:
summarize('PriViLege')

## 10. Ablations on CUB200 (paper Tables 3 and 4)
**Why:** these test the paper's claims about **why** the method works: each loss should add accuracy, and tuning 2 layers should beat tuning more or none.

Table 3 targets (A_Base / A_Last / A_Avg): PKT 79.06 / 70.81 / 73.36; PKT + L_ED 80.31 / 72.70 / 75.04; PKT + L_SKD 82.10 / 73.44 / 76.27.

In [ ]:
for seed in [1, 2, 3, 4, 5]:
    run('cub200', seed, out='abl_PKT',     losses='')
    run('cub200', seed, out='abl_PKT_ED',  losses='-ED')
    run('cub200', seed, out='abl_PKT_SKD', losses='-SKD')
for out in ['abl_PKT', 'abl_PKT_ED', 'abl_PKT_SKD']:
    summarize(out)

Table 4: number of tuned layers, run without the two losses, as in the paper (its 2-layer row equals the PKT row of Table 3). A_Last targets: 0: 60.19, 2: 70.81, 5: 68.52, 7: 63.15, 10: 57.78, 12: 56.02. The B-Prompt stays in layers 0-1; `-taskblock` only changes how many layers are unfrozen.

In [ ]:
for L in [0, 2, 5, 7, 10, 12]:
    run('cub200', seed=1, out=f'layers_{L}', losses='', extra=f'-taskblock {L}')
for L in [0, 2, 5, 7, 10, 12]:
    summarize(f'layers_{L}')

## 11. Free disk space (optional)
Each run leaves ~0.7 GB of `.pth` checkpoints on the Colab disk. Results are already in Drive.

In [ ]:
!find {REPO}/checkpoint -name '*.pth' -delete
!df -h /content | tail -1